In [40]:
import pandas as pd

df = pd.read_csv('../data/raw/ipca_raw.csv', sep=';', encoding='latin1', skiprows=3)

df.head()

,Unnamed: 0,janeiro 2015,fevereiro 2015,marÃ§o 2015,abril 2015,maio 2015,junho 2015,julho 2015,agosto 2015,setembro 2015,...,marÃ§o 2024,abril 2024,maio 2024,junho 2024,julho 2024,agosto 2024,setembro 2024,outubro 2024,novembro 2024,dezembro 2024
0,Brasil,"1,24","1,22","1,32","0,71","0,74","0,79","0,62","0,22","0,54",...,"0,16","0,38","0,46","0,21","0,38","-0,02","0,44","0,56","0,39","0,52"
1,Fonte: IBGE - Ãndice Nacional de PreÃ§os ao C...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Notas,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1 - PerÃ­odo de coleta ajustado ao mÃªs civil ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [41]:
#limpeza e transformação dos dados

# 1. Mantém só a linha do Brasil (linha 0)
df_clean = df.iloc[[0]].copy()

# 2. Remove a coluna desnecessária
df_clean = df_clean.drop(columns=['Unnamed: 0'])

# 3. Transforma colunas em linhas (melt/unpivot)
df_clean = df_clean.melt(var_name='mes_ano', value_name='variacao_pct')

# 4. Corrige o tipo do valor (vírgula → ponto)
df_clean['variacao_pct'] = df_clean['variacao_pct'].str.replace(',', '.').astype(float)

# 5. Converte mes_ano para data
meses_pt = {
    'janeiro': '01', 'fevereiro': '02',
    'março': '03', 'abril': '04',
    'maio': '05', 'junho': '06',
    'julho': '07', 'agosto': '08',
    'setembro': '09', 'outubro': '10',
    'novembro': '11', 'dezembro': '12'
}

def converter_data(texto):
    partes = texto.strip().split(' ')
    mes_texto = partes[0].lower()
    ano = partes[1]

    mes = None
    for chave, valor in meses_pt.items():
        if chave[:3] in mes_texto[:3]:
            mes = valor
            break

    if mes is None:
        print(f"Mês não reconhecido: {repr(texto)}")
        return None
    return pd.to_datetime(f'{ano}-{mes}-01')

df_clean['data'] = df_clean['mes_ano'].apply(converter_data)

# 6. Ordena por data
df_clean = df_clean.sort_values('data').reset_index(drop=True)

# 7. Corrige a grafia de março na coluna mes_ano
df_clean['mes_ano'] = df_clean['mes_ano'].str.replace('marÃ§o', 'março', regex=False)

# Confere se ficou correto
print(df_clean[df_clean['mes_ano'].str.contains('mar')].head(3))

df_clean.head(15)


       mes_ano  variacao_pct       data
2   março 2015          1.32 2015-03-01
14  março 2016          0.43 2016-03-01
26  março 2017          0.25 2017-03-01


,mes_ano,variacao_pct,data
0,janeiro 2015,1.24,2015-01-01
1,fevereiro 2015,1.22,2015-02-01
2,março 2015,1.32,2015-03-01
3,abril 2015,0.71,2015-04-01
4,maio 2015,0.74,2015-05-01
5,junho 2015,0.79,2015-06-01
6,julho 2015,0.62,2015-07-01
7,agosto 2015,0.22,2015-08-01
8,setembro 2015,0.54,2015-09-01
9,outubro 2015,0.82,2015-10-01


In [42]:
# Descobre a grafia exata de março no arquivo
for mes in df_clean['mes_ano'].tolist():
    if 'mar' in mes.lower():
        print(repr(mes))
        break

'março 2015'


In [43]:
#salva os dados limpos na pasta processed
df_clean.to_csv('../data/processed/ipca_limpo.csv', index=False)
print(f"Arquivo Salvo! Total de registros: {len(df_clean)}")

Arquivo Salvo! Total de registros: 120
